In [11]:
using LinearAlgebra
using SparseArrays
using Plots

# =========================================================
# 1. Parameters
# =========================================================
L   = 1.0            # rod length
N   = 200           # number of FV cells
dx  = L / N

E   = 1.0            # Young's modulus
Gc  = 1e-2           # fracture toughness
ℓ   = 0.02/4           # length scale
κ   = 1e-8           # residual stiffness

nsteps = 150*2         # load steps
tol    = 1e-6
maxit  = 50

ū(t) = 0.5 * t     # imposed displacement at right end

# =========================================================
# 2. Grid and fields
# =========================================================
x = collect(dx/2 : dx : L - dx/2)

u = zeros(N)         # displacement
d = zeros(N)         # phase field
H = zeros(N)         # history field

# -----------------------------
# Initial defect as small displacement
# -----------------------------
x0 = 0.5             # defect center
σ  = 0.01            # width of defect

for i in 1:N
    H[i] = 0.2 * exp(-0.5*((x[i]-x0)/σ)^2)
end

# Storage for all steps
u_hist = zeros(N, nsteps)
d_hist = zeros(N, nsteps)
H_hist = zeros(N, nsteps)

ubar_hist = zeros(nsteps)
reaction_hist = zeros(nsteps)

# =========================================================
# 3. Helper functions
# =========================================================
g(d) = (1 - d)^2 + κ
εplus(ε) = max(ε, 0.0)

# =========================================================
# 4. Mechanical solver (FV)
# =========================================================
function solve_mechanics!(u, d, ubar)
    A = zeros(N, N)
    b = zeros(N)

    for i in 1:N
        # left face
        if i == 1
            uL = -u[1]          # u(0)=0
            dL = d[1]
        else
            uL = u[i-1]
            dL = d[i-1]
        end
        εL = (u[i] - uL) / dx
        println(εL > 0)
        cL = E/dx * (εL > 0 ? g(0.5*(d[i]+dL)) : 1.0)

        # right face
        if i == N
            uR = 2*ubar - u[N]   # u(L)=ubar
            dR = d[N]
        else
            uR = u[i+1]
            dR = d[i+1]
        end
        εR = (uR - u[i]) / dx
        println(εR > 0)
        cR = E/dx * (εR > 0 ? g(0.5*(d[i]+dR)) : 1.0)

        # matrix assembly
        A[i,i] += cL + cR
        if i > 1
            A[i,i-1] -= cL
        end
        if i < N
            A[i,i+1] -= cR
        end

        if i == N
            b[i] += cR * 2*ubar
        end
    end

    u[:] = A \ b
    return u, A, b
end

# =========================================================
# 5. Phase-field solver (FV)
# =========================================================
function solve_phasefield!(d, H)
    A = zeros(N, N)
    b = zeros(N)

    for i in 1:N
        A[i,i] = 2*Gc*ℓ/dx^2 + Gc/ℓ + 2*H[i]

        if i > 1
            A[i,i-1] = -Gc*ℓ/dx^2
        else
            A[i,i] -= Gc*ℓ/dx^2   # d'(0)=0
        end

        if i < N
            A[i,i+1] = -Gc*ℓ/dx^2
        else
            A[i,i] -= Gc*ℓ/dx^2   # d'(L)=0
        end

        b[i] = 2*H[i]
    end

    d[:] = A \ b #clamp.(A \ b, 0.0, 1.0)
    return d, A, b
end


solve_phasefield! (generic function with 1 method)

In [ ]:
function dfdx(u, ubar)
    fx = zeros(N)
    
    for i in 1:N
        # left face
        if i == 1
            uL = -u[1]          # u(0)=0
        else
            uL = u[i-1]
        end
        εL = (u[i] - uL) / dx
        println(εL>0)
    
        # right face
        if i == N
            uR = 2*ubar - u[N]   # u(L)=ubar
        else
            uR = u[i+1]
        end
        εR = (uR - u[i]) / dx
        println(εR>0)
    end
end

dfdx (generic function with 1 method)

In [23]:
# strain u_x (with variable E)
function DuDx(u)
    idx = 1/dx
    u_x = zeros(N)

    # forward FD left
    u_x[1] = idx*(u[2]-u[1])

    # centred FD interior
    for i in 2:N-1
        u_x[i] = 0.5*idx*(u[i+1]-u[i-1])
    end

    # Neumann right
    u_x[N] = idx*(u[N-1]-u[N])

    return u_x
end

DuDx (generic function with 1 method)

In [37]:
# midpoint
function stagav(f::Vector)
    N = length(f)
    f_mid = zeros(N-1)
    for i in 1:N-1
        f_mid[i] = 0.5*(f[i]+f[i+1])
    end
    return f_mid
end

# solver for a single timestep
function Abu_mine(u, d, ubar)
    idx = 1/dx
    u_x = DuDx(u)
    

        # Step 3: displacement
        g = (1.0 .- d).^2 .+ κ
        g_mid = stagav(g)

        A = zeros(N,N)
        b = zeros(N)

        # Dirichlet left
        A[1,1] = 1.0
        b[1] = 0.0

        for i in 2:N-1
            if u_x[i] > 0
                A[i,i-1] = g_mid[i-1]*E*idx
                A[i,i]   = -E*idx*(g_mid[i]+g_mid[i-1])
                A[i,i+1] = g_mid[i]*E*idx
            else
                A[i,i-1] = idx
                A[i,i]   = -2*idx
                A[i,i+1] = idx
            end
        end

        # Dirichlet
        A[N,N] = 1.0
        b[N] = ubar

    return A, b
end 


Abu_mine (generic function with 1 method)

In [24]:
function Abu_auto(u, d, ubar)
    A = zeros(N, N)
    b = zeros(N)

    for i in 1:N
        # left face
        if i == 1
            uL = -u[1]          # u(0)=0
            dL = d[1]
        else
            uL = u[i-1]
            dL = d[i-1]
        end
        εL = (u[i] - uL) / dx
        cL = E/dx * (εL > 0 ? g(0.5*(d[i]+dL)) : 1.0)

        # right face
        if i == N
            uR = 2*ubar - u[N]   # u(L)=ubar
            dR = d[N]
        else
            uR = u[i+1]
            dR = d[i+1]
        end
        εR = (uR - u[i]) / dx
        cR = E/dx * (εR > 0 ? g(0.5*(d[i]+dR)) : 1.0)

        # matrix assembly
        A[i,i] += cL + cR
        if i > 1
            A[i,i-1] -= cL
        end
        if i < N
            A[i,i+1] -= cR
        end

        if i == N
            b[i] += ubar#cR * 2*ubar
        end
    end
    A[1,1] = 1.0
    A[1,2] = 0.0
    A[N,N] = 1.0
    A[N,N-1] = 0.0
    
    return A, b
end

Abu_auto (generic function with 1 method)

In [25]:
Au, bu = Abu_auto(u,d,ubar)

([1.0 0.0 … 0.0 0.0; -200.0 400.0 … 0.0 0.0; … ; 0.0 0.0 … 400.0 -200.0; 0.0 0.0 … 0.0 1.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0016666666666666668])

In [26]:
Au

200×200 Matrix{Float64}:
    1.0     0.0     0.0     0.0     0.0  …     0.0     0.0     0.0     0.0
 -200.0   400.0  -200.0     0.0     0.0        0.0     0.0     0.0     0.0
    0.0  -200.0   400.0  -200.0     0.0        0.0     0.0     0.0     0.0
    0.0     0.0  -200.0   400.0  -200.0        0.0     0.0     0.0     0.0
    0.0     0.0     0.0  -200.0   400.0        0.0     0.0     0.0     0.0
    0.0     0.0     0.0     0.0  -200.0  …     0.0     0.0     0.0     0.0
    0.0     0.0     0.0     0.0     0.0        0.0     0.0     0.0     0.0
    0.0     0.0     0.0     0.0     0.0        0.0     0.0     0.0     0.0
    0.0     0.0     0.0     0.0     0.0        0.0     0.0     0.0     0.0
    0.0     0.0     0.0     0.0     0.0        0.0     0.0     0.0     0.0
    0.0     0.0     0.0     0.0     0.0  …     0.0     0.0     0.0     0.0
    0.0     0.0     0.0     0.0     0.0        0.0     0.0     0.0     0.0
    0.0     0.0     0.0     0.0     0.0        0.0     0.0     0.0     0.0


In [38]:
my_Au, my_bu = Abu_mine(u,d,ubar)

([1.0 0.0 … 0.0 0.0; 200.0 -400.0 … 0.0 0.0; … ; 0.0 0.0 … -400.0 200.0; 0.0 0.0 … 0.0 1.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0016666666666666668])

In [39]:
my_Au

200×200 Matrix{Float64}:
   1.0     0.0     0.0     0.0     0.0  …     0.0     0.0     0.0    0.0
 200.0  -400.0   200.0     0.0     0.0        0.0     0.0     0.0    0.0
   0.0   200.0  -400.0   200.0     0.0        0.0     0.0     0.0    0.0
   0.0     0.0   200.0  -400.0   200.0        0.0     0.0     0.0    0.0
   0.0     0.0     0.0   200.0  -400.0        0.0     0.0     0.0    0.0
   0.0     0.0     0.0     0.0   200.0  …     0.0     0.0     0.0    0.0
   0.0     0.0     0.0     0.0     0.0        0.0     0.0     0.0    0.0
   0.0     0.0     0.0     0.0     0.0        0.0     0.0     0.0    0.0
   0.0     0.0     0.0     0.0     0.0        0.0     0.0     0.0    0.0
   0.0     0.0     0.0     0.0     0.0        0.0     0.0     0.0    0.0
   0.0     0.0     0.0     0.0     0.0  …     0.0     0.0     0.0    0.0
   0.0     0.0     0.0     0.0     0.0        0.0     0.0     0.0    0.0
   0.0     0.0     0.0     0.0     0.0        0.0     0.0     0.0    0.0
   ⋮                      

In [46]:
my_u = my_Au \ my_bu

200-element Vector{Float64}:
 -4.066662100381951e-16
  8.37520937983031e-6
  1.6750418760067286e-5
  2.5125628140304264e-5
  3.350083752054124e-5
  4.187604690077822e-5
  5.02512562810152e-5
  5.862646566125217e-5
  6.700167504148914e-5
  7.537688442172611e-5
  8.375209380196308e-5
  9.212730318220004e-5
  0.000100502512562437
  ⋮
  0.0015745393634840766
  0.0015829145728643115
  0.0015912897822445466
  0.0015996649916247816
  0.001608040201005017
  0.0016164154103852524
  0.001624790619765488
  0.0016331658291457234
  0.0016415410385259592
  0.0016499162479061949
  0.0016582914572864308
  0.0016666666666666668

In [47]:
auto_u = Au \ bu

200-element Vector{Float64}:
 -4.066662100381951e-16
  8.37520937983031e-6
  1.6750418760067286e-5
  2.5125628140304264e-5
  3.350083752054124e-5
  4.187604690077822e-5
  5.02512562810152e-5
  5.862646566125217e-5
  6.700167504148914e-5
  7.537688442172611e-5
  8.375209380196308e-5
  9.212730318220004e-5
  0.000100502512562437
  ⋮
  0.0015745393634840766
  0.0015829145728643115
  0.0015912897822445466
  0.0015996649916247816
  0.001608040201005017
  0.0016164154103852524
  0.001624790619765488
  0.0016331658291457234
  0.0016415410385259592
  0.0016499162479061949
  0.0016582914572864308
  0.0016666666666666668

In [50]:
maximum(my_u .- auto_u)

0.0